In [1]:
import pandas as pd
import json
from pydantic import BaseModel
from openai import OpenAI

## Implementation using MockLLM

In [15]:
# MockLLM server needs to be started for testing
#  mockllm start --responses mock_responses.yml --port 3000   

In [3]:
# --- Subtask 1.2: Strict JSON Schema Enforcement ---
class AgentOutput(BaseModel):
    recommended_plan_id: str
    discount_applied_flag: bool
    agent_reasoning: str

In [4]:
# --- Subtask 1.2: Tool/Function to query mock DB ---
def query_customer_db(user_id: str) -> dict:
    """Simulates the Usage Analyst role querying the database."""
    logs = pd.read_csv('data/tower_logs.csv')
    subs = pd.read_csv('data/customer_subscriptions.csv')
    transcripts = pd.read_csv('data/transcripts.csv')
    
    user_log = logs[logs['user_id'] == user_id].iloc[0]
    user_sub = subs[subs['user_id'] == user_id].iloc[0]
    user_transcripts = transcripts[transcripts['user_id'] == user_id]['text_transcript'].tolist()
    
    return {
        "usage_gb": user_log['avg_data_gb'],
        "current_plan": user_sub['current_plan_id'],
        "dropped_calls": user_log['dropped_calls'],
        "complaints": user_transcripts
    }

In [5]:

# --- Subtask 1.1 & 1.3: Agentic Workflow & Guardrail ---
def run_agentic_workflow(user_id: str) -> str:
    """Executes the State Machine pipeline."""
    
    # 1. Usage Analyst State
    customer_data = query_customer_db(user_id)
    
    # 2. Plan Advisor & Retention Specialist State (Simulated via MockLLM)
    prompt = f"Analyze customer {user_id}. Data: {customer_data}. Output JSON."
    
    # Point the standard OpenAI client to our local MockLLM server
    client = OpenAI(base_url="http://localhost:3000/v1", api_key="mock-key")
    response = client.chat.completions.create(
        model="mock-model",
        messages=[{"role": "user", "content": prompt}]
    )
    
    # Parse the deterministic JSON response from our mock_responses.yml
    raw_output = response.choices[0].message.content
    llm_decision = json.loads(raw_output)
    
    # 3. Validation Guardrail State (Subtask 1.3)
    proposed_discount = llm_decision.get("proposed_discount_amount", 0)
    if proposed_discount > 20:
        llm_decision["agent_reasoning"] += f" [GUARDRAIL TRIGGERED: Discount reduced from ${proposed_discount} to max $20.]"
        llm_decision["discount_applied_flag"] = True
    
    # Clean up the dictionary before Pydantic validation
    llm_decision.pop("proposed_discount_amount", None)
    
    # Enforce strictly via Pydantic and return JSON string
    final_output = AgentOutput(**llm_decision)
    return final_output.model_dump_json(indent=2)

In [7]:
# Execute test for MVP
print(run_agentic_workflow("CUST_002"))

{
  "recommended_plan_id": "P_ULTRA",
  "discount_applied_flag": true,
  "agent_reasoning": "Customer is highly frustrated with dropped calls. Recommending Ultra plan and offering a $50 discount to prevent churn. [GUARDRAIL TRIGGERED: Discount reduced from $50 to max $20.]"
}


## implementation using a custome mock llm

In [8]:
import pandas as pd
import json
from pydantic import BaseModel

In [9]:
# --- Subtask 1.2: Strict JSON Schema Enforcement ---
class AgentOutput(BaseModel):
    recommended_plan_id: str
    discount_applied_flag: bool
    final_discount_amount: float 
    agent_reasoning: str

# --- Subtask 1.2: Tool/Function to query mock DB ---
def query_customer_db(user_id: str) -> dict:
    """Simulates the Usage Analyst role querying the database."""
    logs = pd.read_csv('data/tower_logs.csv')
    subs = pd.read_csv('data/customer_subscriptions.csv')
    transcripts = pd.read_csv('data/transcripts.csv')
    
    user_log = logs[logs['user_id'] == user_id].iloc[0]
    user_sub = subs[subs['user_id'] == user_id].iloc[0]
    user_transcripts = transcripts[transcripts['user_id'] == user_id]['text_transcript'].tolist()
    
    return {
        "usage_gb": user_log['avg_data_gb'],
        "current_plan": user_sub['current_plan_id'],
        "dropped_calls": user_log['dropped_calls'],
        "complaints": user_transcripts
    }

# --- Dynamic Mock LLM (Simulating Advisor & Retention Specialist) ---
class CustomMockLLM:
    def generate_response(self, customer_data: dict) -> dict:
        """Simulates LLM dynamically reasoning based on live data."""
        dropped_calls = customer_data.get('dropped_calls', 0)
        
        # Dynamic Plan Upgrade
        recommended_plan = "P_ULTRA" 
        
        # Dynamic Discount: $15 penalty per dropped call
        proposed_discount = dropped_calls * 15.0
        apply_discount = proposed_discount > 0
        
        reasoning = f"Analyzed usage. Customer had {dropped_calls} dropped calls. "
        if apply_discount:
            reasoning += f"Proposing dynamic discount of ${proposed_discount}."
        else:
            reasoning += "Service stable. No discount needed."
            
        return {
            "recommended_plan_id": recommended_plan,
            "discount_applied_flag": apply_discount,
            "proposed_discount_amount": proposed_discount,
            "agent_reasoning": reasoning
        }

# --- Subtask 1.1 & 1.3: Agentic Workflow & Guardrail ---
def run_agentic_workflow(user_id: str) -> str:
    """Executes the State Machine pipeline."""
    # 1. Usage Analyst State
    customer_data = query_customer_db(user_id)
    
    # 2. Plan Advisor & Retention Specialist State
    llm = CustomMockLLM()
    llm_decision = llm.generate_response(customer_data)
    
    # 3. Validation Guardrail State (Subtask 1.3)
    proposed_discount = llm_decision.get("proposed_discount_amount", 0.0)
    final_discount = proposed_discount
    
    # Guardrail intercepts and caps at $20
    if proposed_discount > 20.0:
        final_discount = 20.0
        llm_decision["agent_reasoning"] += f" [GUARDRAIL TRIGGERED: Discount reduced from ${proposed_discount} to max $20.]"
    
    # Finalize state for Pydantic
    llm_decision["final_discount_amount"] = final_discount
    llm_decision.pop("proposed_discount_amount", None)
    
    # Enforce strictly via Pydantic and return JSON string
    final_output = AgentOutput(**llm_decision)
    return final_output.model_dump_json(indent=2)

In [10]:
print("Testing CUST_002:")
print(run_agentic_workflow("CUST_002"))

Testing CUST_002:
{
  "recommended_plan_id": "P_ULTRA",
  "discount_applied_flag": true,
  "final_discount_amount": 20.0,
  "agent_reasoning": "Analyzed usage. Customer had 2 dropped calls. Proposing dynamic discount of $30.0. [GUARDRAIL TRIGGERED: Discount reduced from $30.0 to max $20.]"
}


In [12]:
print("\nTesting CUST_013:")
print(run_agentic_workflow("CUST_013"))


Testing CUST_013:
{
  "recommended_plan_id": "P_ULTRA",
  "discount_applied_flag": false,
  "final_discount_amount": 0.0,
  "agent_reasoning": "Analyzed usage. Customer had 0 dropped calls. Service stable. No discount needed."
}


In [13]:
print("\nTesting CUST_376:")
print(run_agentic_workflow("CUST_376"))


Testing CUST_376:
{
  "recommended_plan_id": "P_ULTRA",
  "discount_applied_flag": true,
  "final_discount_amount": 15.0,
  "agent_reasoning": "Analyzed usage. Customer had 1 dropped calls. Proposing dynamic discount of $15.0."
}
